# Kaon Flux Fix — OLD vs NEW checkout comparison (Run 4b BNB $\nu$ overlay)

We found (see `spline_and_systematic_shift_checks.ipynb`) that all three kaon PrimaryHadron
multisigma "spline" knobs were broken in the eventweight production:

- **`kminus_PrimaryHadronNormalization`**: out-of-bounds heap read → garbage weights up to ~1e243, with the $\sigma$ alignment destroyed (`[1,2,3,4,garbage...]`).
- **`kplus_PrimaryHadronFeynmanScaling`**: all 7 universes bit-identical per event → zero uncertainty content.
- **`kzero_PrimaryHadronSanfordWang`**: all universes = −9898 sentinel → zero information.

The ubsim calculators were fixed and a Run 4b $\nu$ overlay test sample was re-processed. In the
new production, **kminus** stays a multisigma knob (now `multisigma [-1,0,1,2,3]`, so the
normalization weights should be exactly $w = 1+\sigma = [0,1,2,3,4]$ for K⁻-parent events), while
**kplus** and **kzero** became **1000-universe multisims** (they sample a fit-parameter covariance
matrix, so multisim throws are the natural mode — a spline in $\sigma$ was never meaningful).

| | file |
|---|---|
| **OLD** | `data_files/checkout_MCC9.10_Run4b_v10_04_07_20_BNB_nu_overlay_retuple_retuple_hist.root` (full Run 4b production, broken kaon knobs) |
| **NEW** | `other_files/run_4b_nu_overlay_splines_kaon_flux_fix.root` (test reprocessing with fixed calculators) |

Checks performed here:
1. file/tree structure and per-knob universe counts (only the 3 kaon knobs should change);
2. event overlap, within-file tree alignment, duplicates, POT bookkeeping;
3. everything that should NOT have changed (non-kaon knobs, CV weights, reco $E_\nu$) is bit-identical on matched events;
4. kminus fix is exactly the analytic $w=1+\sigma$ pattern, on exactly the K⁻-parent events;
5. kplus/kzero multisims are healthy (finite, no sentinels, sensible spread, on exactly the K-parent events);
6. impact on the reconstructed $E_\nu$ flux uncertainty, old vs new;
7. $\nu_e$-specific weights and uncertainties — the $\nu_e$ flux is kaon-dominated above
   ~1 GeV, so this is where the fix matters most.

Every check is recorded and summarized at the bottom as PASS/WARN.


In [ ]:
import gc, warnings
import numpy as np
import awkward as ak
import uproot
import polars as pl
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 110

OLD_PATH = "/nevis/riverside/data/leehagaman/ngem/data_files/checkout_MCC9.10_Run4b_v10_04_07_20_BNB_nu_overlay_retuple_retuple_hist.root"
NEW_PATH = "/nevis/riverside/data/leehagaman/ngem/other_files/run_4b_nu_overlay_splines_kaon_flux_fix.root"
NEW_PATH = "/nevis/riverside/data/leehagaman/ngem/other_files/test_spline.root"

KMINUS = "kminus_PrimaryHadronNormalization"
KPLUS  = "kplus_PrimaryHadronFeynmanScaling"
KZERO  = "kzero_PrimaryHadronSanfordWang"
KAON_KNOBS = [KMINUS, KPLUS, KZERO]

CAP  = 30.0                      # same convention as src/systematics.py / postprocessing.py
BINS = np.linspace(0, 2000, 21)  # MeV, reco Enu
NB   = len(BINS) - 1
BINC = 0.5 * (BINS[:-1] + BINS[1:])

C_OLD, C_NEW = "#d95f02", "#1f77b4"   # fixed colorblind-safe pair: orange=OLD, blue=NEW

f_old = uproot.open(OLD_PATH)
f_new = uproot.open(NEW_PATH)
sw_old, sw_new = f_old["spline_weights"], f_new["spline_weights"]
N_old, N_new = sw_old.num_entries, sw_new.num_entries
print(f"OLD: {N_old:,} events   NEW: {N_new:,} events")

checks = []
def record(name, ok, detail=""):
    checks.append((name, "PASS" if ok else "WARN", detail))
    print(f"[{'PASS' if ok else 'WARN'}] {name}" + (f" — {detail}" if detail else ""))

def cap_weights(w):
    # nan/inf -> 1, >30 -> 1, <0 -> 1 (0 is kept), matching src/systematics.py
    w = np.nan_to_num(w, nan=1.0, posinf=1.0, neginf=1.0)
    return np.where((w > CAP) | (w < 0), 1.0, w)

def load_knob_matrix(tree, branch, entry_stop=None):
    # rectangular (n_events, n_universes) float64 matrix for one weight branch
    a = tree[branch].array(entry_stop=entry_stop, library="ak")
    n = ak.to_numpy(ak.num(a))
    assert n.min() == n.max(), f"{branch}: ragged universe counts {np.unique(n)}"
    return np.asarray(ak.flatten(a)).reshape(len(n), n[0])


## 1. File / tree structure

The NEW file is a raw checkout of the test reprocessing, so some auxiliary trees/histograms differ
(that is expected and harmless); what matters is `spline_weights` and the per-knob universe counts.
Only the three kaon knobs should have changed: kminus 7→5, kplus 7→1000, kzero 7→1000.

In [ ]:
def tree_summary(f):
    return {k: f[k].num_entries for k in f.keys(recursive=True, cycle=False)
            if hasattr(f[k], "num_entries")}

ts_old, ts_new = tree_summary(f_old), tree_summary(f_new)
print(f"{'tree':45s} {'OLD':>12s} {'NEW':>12s}")
for k in sorted(set(ts_old) | set(ts_new)):
    so = f"{ts_old[k]:,}" if k in ts_old else "--"
    sn = f"{ts_new[k]:,}" if k in ts_new else "--"
    print(f"{k:45s} {so:>12s} {sn:>12s}")

br_old, br_new = set(sw_old.keys()), set(sw_new.keys())
print("\nspline_weights branches only in OLD:", sorted(br_old - br_new))
print("spline_weights branches only in NEW:", sorted(br_new - br_old))
record("spline_weights branch set unchanged (only 'samdef' metadata dropped)",
       (br_old - br_new) <= {"samdef"} and not (br_new - br_old))

# universe counts, probed on the first 2000 events of each file
COMMON = sorted((br_old & br_new) - {"run", "subrun", "event", "entry"})
probe_old = sw_old.arrays(COMMON, entry_stop=2000, library="ak")
probe_new = sw_new.arrays(COMMON, entry_stop=2000, library="ak")
nuniv = {}
for c in COMMON:
    no = np.unique(ak.to_numpy(ak.num(probe_old[c])))
    nn = np.unique(ak.to_numpy(ak.num(probe_new[c])))
    nuniv[c] = (no, nn)
changed = {c: v for c, v in nuniv.items() if not np.array_equal(v[0], v[1])}
print(f"\n{len(COMMON)} common weight branches; universe counts changed for:")
for c, (no, nn) in sorted(changed.items()):
    print(f"  {c:45s} {list(no)} -> {list(nn)}")
record("only the 3 kaon knobs changed universe count", set(changed) == set(KAON_KNOBS),
       f"changed: {sorted(changed)}")
record("kminus 7->5, kplus 7->1000, kzero 7->1000",
       all(c in changed for c in KAON_KNOBS)
       and list(changed.get(KMINUS, ([], []))[1]) == [5]
       and list(changed.get(KPLUS, ([], []))[1]) == [1000]
       and list(changed.get(KZERO, ([], []))[1]) == [1000])
del probe_old, probe_new; gc.collect();


## 2. Event overlap, tree alignment, duplicates, POT

The NEW file is a *test* reprocessing, so it need not contain every OLD event; but shared
(run, subrun) pairs should contain the same events, RSE must be unique within each file, and
the trees we read row-aligned must actually be row-aligned.

In [ ]:
def rse_df(tree):
    a = tree.arrays(["run", "subrun", "event"], library="np")
    return pl.DataFrame({k: a[k].astype(np.int64) for k in ("run", "subrun", "event")})

rse_old, rse_new = rse_df(sw_old), rse_df(sw_new)

# within-file row alignment of the trees we will index side by side
for tag, f, r in [("OLD", f_old, rse_old), ("NEW", f_new, rse_new)]:
    for tname in ["wcpselection/T_eval", "singlephotonana/eventweight_tree"]:
        record(f"{tag}: spline_weights row-aligned with {tname}", r.equals(rse_df(f[tname])))

for tag, r in [("OLD", rse_old), ("NEW", rse_new)]:
    ndup = r.height - r.unique().height
    record(f"{tag}: run/subrun/event unique", ndup == 0, f"{ndup} duplicates")

m = (rse_old.with_row_index("i_old")
     .join(rse_new.with_row_index("i_new"), on=["run", "subrun", "event"], how="inner")
     .sort("i_old"))
idx_old = m["i_old"].to_numpy().astype(np.int64)
idx_new = m["i_new"].to_numpy().astype(np.int64)
print(f"\nmatched events: {len(idx_old):,}  "
      f"(OLD-only {N_old - len(idx_old):,}, NEW-only {N_new - len(idx_new):,})")
record("every NEW event also exists in OLD", len(idx_new) == N_new,
       f"{N_new - len(idx_new):,} NEW-only events")

# shared subruns should contain identical event counts
cnt_old = rse_old.group_by(["run", "subrun"]).len().rename({"len": "n_old"})
cnt_new = rse_new.group_by(["run", "subrun"]).len().rename({"len": "n_new"})
cc = cnt_old.join(cnt_new, on=["run", "subrun"], how="inner")
mism = cc.filter(pl.col("n_old") != pl.col("n_new"))
print(f"shared (run,subrun): {cc.height:,} of OLD {cnt_old.height:,} / NEW {cnt_new.height:,}")
record("shared subruns have identical event counts", mism.height == 0,
       f"{mism.height}/{cc.height} shared subruns differ")
if mism.height:
    print(mism.head(10))

# POT bookkeeping
pots = {}
for tag, f in [("OLD", f_old), ("NEW", f_new)]:
    p = f["wcpselection/T_pot"].arrays(["pot_tor875", "pot_tor875good"], library="np")
    pots[tag] = (p["pot_tor875"].sum(), p["pot_tor875good"].sum())
    print(f"{tag}: sum pot_tor875 = {pots[tag][0]:.4e}   pot_tor875good = {pots[tag][1]:.4e}")
pot_ratio, evt_ratio = pots["NEW"][0] / pots["OLD"][0], N_new / N_old
print(f"NEW/OLD POT ratio = {pot_ratio:.4f}   event-count ratio = {evt_ratio:.4f}")
record("NEW/OLD POT ratio consistent with event-count ratio (within 5%)",
       abs(pot_ratio / evt_ratio - 1) < 0.05,
       f"POT ratio {pot_ratio:.4f} vs event ratio {evt_ratio:.4f}")


### 2b. Deeper POT / event bookkeeping

The three WARNs above share one cause, diagnosed here: the OLD production contains T_pot
subruns that have **no events** in the file, while the NEW test sample recovers them.

In [ ]:
for tag, f in [("OLD", f_old), ("NEW", f_new)]:
    p = f["wcpselection/T_pot"].arrays(["runNo", "subRunNo", "pot_tor875"], library="np")
    pot = pl.DataFrame({"run": p["runNo"].astype(np.int64),
                        "subrun": p["subRunNo"].astype(np.int64), "pot": p["pot_tor875"]})
    ev_sr = (rse_old if tag == "OLD" else rse_new).select(["run", "subrun"]).unique()
    ndup = pot.height - pot.select(["run", "subrun"]).unique().height
    extra_pot = (pot.filter(pl.struct(["run", "subrun"]).is_duplicated())
                 .group_by(["run", "subrun"]).agg(pl.col("pot").sum() - pl.col("pot").first())
                 ["pot"].sum()) if ndup else 0.0
    pot_u = pot.group_by(["run", "subrun"]).agg(pl.col("pot").sum())
    eventless = pot_u.join(ev_sr, on=["run", "subrun"], how="anti")
    tot = pot_u["pot"].sum()
    nev = N_old if tag == "OLD" else N_new
    frac = eventless["pot"].sum() / tot
    print(f"{tag}: T_pot subruns {pot_u.height:,} (duplicate rows {ndup}, "
          f"extra POT in duplicates {extra_pot:.3e}), event subruns {ev_sr.height:,}")
    print(f"  T_pot subruns with ZERO events: {eventless.height:,}, carrying "
          f"{eventless['pot'].sum():.4e} POT = {frac:.2%} of total")
    print(f"  events per 1e19 POT: {nev / tot * 1e19:.1f}")
    record(f"{tag}: T_pot subruns all have events (POT/event bookkeeping consistent)",
           frac < 0.005, f"{eventless.height:,} event-less subruns carry {frac:.2%} of the POT")

# where do the NEW-only events live?
new_only = rse_new.join(rse_old, on=["run", "subrun", "event"], how="anti")
old_sr = rse_old.select(["run", "subrun"]).unique()
n_in_shared = new_only.join(old_sr, on=["run", "subrun"], how="semi").height
absent_sr = new_only.join(old_sr, on=["run", "subrun"], how="anti").select(["run", "subrun"]).unique()
p_old = f_old["wcpselection/T_pot"].arrays(["runNo", "subRunNo"], library="np")
pot_sr_old = pl.DataFrame({"run": p_old["runNo"].astype(np.int64),
                           "subrun": p_old["subRunNo"].astype(np.int64)}).unique()
n_in_old_pot = absent_sr.join(pot_sr_old, on=["run", "subrun"], how="semi").height
print(f"\nNEW-only events: {new_only.height:,} -- {n_in_shared:,} inside subruns OLD also has events "
      f"for, {new_only.height - n_in_shared:,} in {absent_sr.height:,} subruns with no OLD events at all")
print(f"of those {absent_sr.height:,} subruns, {n_in_old_pot:,} nonetheless appear in the OLD T_pot")
print("=> the OLD production counted POT for subruns whose events never made it into the file; "
      "the NEW test sample recovers (most of) them.")


## 3. Everything that should NOT have changed

Only the eventweight kaon calculators were fixed, so on matched events:
- every non-kaon spline knob should be **bit-identical** (multisigma knobs are deterministic);
- `T_eval` `weight_cv`, `weight_spline` and `T_KINEvars` `kine_reco_Enu` should be identical;
- `weightsReint` (1000 random Geant4-reinteraction throws) is only *statistically* reproducible
  unless the random seeds were preserved.

In [ ]:
# CV weights and reco Enu on matched events
ev_old = f_old["wcpselection/T_eval"].arrays(["weight_cv", "weight_spline"], library="np")
ev_new = f_new["wcpselection/T_eval"].arrays(["weight_cv", "weight_spline"], library="np")
enu_old = f_old["wcpselection/T_KINEvars"]["kine_reco_Enu"].array(library="np")
enu_new = f_new["wcpselection/T_KINEvars"]["kine_reco_Enu"].array(library="np")

for name, a, b in [("T_eval weight_cv", ev_old["weight_cv"], ev_new["weight_cv"]),
                   ("T_eval weight_spline", ev_old["weight_spline"], ev_new["weight_spline"]),
                   ("T_KINEvars kine_reco_Enu", enu_old, enu_new)]:
    x, y = np.asarray(a)[idx_old], np.asarray(b)[idx_new]
    eq = (x == y) | (np.isnan(x) & np.isnan(y))
    record(f"matched events: {name} identical", eq.all(),
           f"{(~eq).sum():,}/{len(eq):,} differ" if not eq.all() else "")
    if not eq.all() and (~eq).sum() <= 30:
        for xa, xb in zip(x[~eq], y[~eq]):
            print(f"    {xa!r} -> {xb!r}")


In [ ]:
# every non-kaon small knob must be bit-identical on matched events
SMALL = [c for c in COMMON if c not in KAON_KNOBS and nuniv[c][0].max() <= 50]
rows = []
for i, c in enumerate(SMALL):
    A = load_knob_matrix(sw_old, c)
    B = load_knob_matrix(sw_new, c)
    a, b = A[idx_old], B[idx_new]
    eq = (a == b) | (np.isnan(a) & np.isnan(b))
    frac_eq = eq.mean()
    with np.errstate(invalid="ignore"):
        d = np.abs(a - b)
    d = d[np.isfinite(d)]
    rows.append((c, frac_eq, float(d.max()) if d.size else 0.0))
    del A, B, a, b, eq
    if (i + 1) % 10 == 0:
        print(f"  ...{i+1}/{len(SMALL)} knobs compared"); gc.collect()

rows.sort(key=lambda r: r[1])
bad = [r for r in rows if r[1] < 1.0]
print(f"\n{len(SMALL)} non-kaon knobs compared on {len(idx_old):,} matched events; "
      f"{len(SMALL) - len(bad)} bit-identical")
for c, fe, mx in bad[:15]:
    print(f"  NOT identical: {c:45s} frac_equal={fe:.6f}  max|diff|={mx:.3e}")
record("all non-kaon spline knobs bit-identical on matched events", not bad,
       f"{len(bad)} knobs differ" if bad else "")
gc.collect();


In [ ]:
# weightsReint: random throws -- identical only if seeds were preserved. NOTE: the checkout
# spline_weights tree does not carry weightsReint (create_rw_syst_df.py merges it in later from
# the reint eventweights), so this comparison only runs if the branch exists in both files.
if "weightsReint" in br_old and "weightsReint" in br_new:
    NS = 60000
    Wo = load_knob_matrix(sw_old, "weightsReint", entry_stop=NS)
    Wn = load_knob_matrix(sw_new, "weightsReint", entry_stop=NS)
    sel = (idx_old < NS) & (idx_new < NS)
    a, b = Wo[idx_old[sel]], Wn[idx_new[sel]]
    row_ident = np.all((a == b) | (np.isnan(a) & np.isnan(b)), axis=1)
    print(f"weightsReint on {sel.sum():,} matched events: "
          f"{row_ident.mean():.1%} rows bit-identical")

    ma, mb = np.nanmean(a, axis=1), np.nanmean(b, axis=1)
    sa, sb = np.nanstd(a, axis=1), np.nanstd(b, axis=1)
    if row_ident.all():
        record("weightsReint identical (seeds preserved)", True)
    else:
        # different seeds are fine; the per-event mean/spread must still agree statistically
        dm = abs(np.mean(ma) - np.mean(mb))
        vary = sa > 0
        ratio = np.mean(sb[vary]) / np.mean(sa[vary]) if vary.any() else 1.0
        record("weightsReint statistically consistent (new seeds)",
               dm < 0.01 and abs(ratio - 1) < 0.10,
               f"mean diff {dm:.4f}, avg per-event std ratio NEW/OLD {ratio:.3f}")
        fig, axs = plt.subplots(1, 2, figsize=(10, 3.5))
        bins_m = np.linspace(0.5, 1.5, 60)
        axs[0].hist(ma, bins=bins_m, histtype="step", color=C_OLD, label="OLD")
        axs[0].hist(mb, bins=bins_m, histtype="step", color=C_NEW, label="NEW")
        axs[0].set_xlabel("per-event mean of weightsReint"); axs[0].legend(); axs[0].set_yscale("log")
        bins_s = np.linspace(0, 1.0, 60)
        axs[1].hist(sa, bins=bins_s, histtype="step", color=C_OLD, label="OLD")
        axs[1].hist(sb, bins=bins_s, histtype="step", color=C_NEW, label="NEW")
        axs[1].set_xlabel("per-event std of weightsReint"); axs[1].legend(); axs[1].set_yscale("log")
        fig.suptitle("weightsReint sanity (matched events, first 60k rows)")
        plt.tight_layout(); plt.show()
    del Wo, Wn, a, b; gc.collect()
else:
    print("weightsReint not present in the checkout spline_weights trees "
          "(merged in later by create_rw_syst_df.py) -- nothing to compare at this stage.")
    record("weightsReint comparison", True, "skipped: branch not in checkout spline trees")


## 4. kminus fix validation

Expected NEW behavior: `multisigma [-1, 0, +1, +2, +3]`, normalization weight $w = 1+\sigma$,
so K⁻-parent events carry exactly `[0, 1, 2, 3, 4]` and every other event `[1, 1, 1, 1, 1]`.
Cross-checked against the kaon parentage stored in `eventweight_tree` (`MCFlux_ptype` /
`MCFlux_tptype`), and against which events the OLD file flagged.

In [ ]:
KM_new = load_knob_matrix(sw_new, KMINUS)
KM_old = load_knob_matrix(sw_old, KMINUS)

record("kminus NEW: all weights finite", bool(np.isfinite(KM_new).all()))
record("kminus NEW: no -9898 sentinels", not np.any(KM_new < -1000))
with np.errstate(invalid="ignore"):
    record("kminus NEW: no garbage weights (max <= 4)", float(np.nanmax(KM_new)) <= 4.0,
           f"max = {np.nanmax(KM_new):.6g}")
    print(f"OLD kminus for contrast: max weight = {np.nanmax(KM_old):.3e}, "
          f"{int((KM_old > CAP).sum()):,} weights > {CAP}")

varying_new = ~np.all(KM_new == 1.0, axis=1)
varying_old = ~np.all(KM_old == 1.0, axis=1)
print(f"\nNEW varying (K- parent) events: {varying_new.sum():,} / {N_new:,}")
print(f"OLD varying events:              {varying_old.sum():,} / {N_old:,}")

uniq = np.unique(KM_new[varying_new], axis=0) if varying_new.any() else np.empty((0, 5))
print("unique NEW varying patterns:", uniq)
expect = np.array([0.0, 1.0, 2.0, 3.0, 4.0])
record("kminus NEW: varying events exactly w=1+sigma = [0,1,2,3,4]",
       uniq.shape == (1, 5) and np.array_equal(uniq[0], expect),
       f"patterns found: {uniq[:5]}")
record("kminus NEW: sigma=0 universe (index 1) == 1 for every event",
       bool(np.all(KM_new[:, 1] == 1.0)))
record("kminus: same events flagged K- in OLD and NEW (matched events)",
       bool(np.array_equal(varying_old[idx_old], varying_new[idx_new])),
       f"OLD-matched {varying_old[idx_old].sum():,} vs NEW-matched {varying_new[idx_new].sum():,}")

# kaon parentage cross-check (works for GEANT or PDG codes; determined empirically)
fx_new = f_new["singlephotonana/eventweight_tree"].arrays(
    ["MCFlux_ptype", "MCFlux_tptype"], library="np")

def parent_crosscheck(varying, knob_label):
    ok_any = False
    for br in ["MCFlux_tptype", "MCFlux_ptype"]:
        codes = fx_new[br]
        vcodes = np.unique(codes[varying]) if varying.any() else np.array([])
        # codes whose events are >50% varying = the kaon code(s) for this knob
        kcodes = [c for c in vcodes if varying[codes == c].mean() > 0.5]
        predicted = np.isin(codes, kcodes)
        exact = np.array_equal(predicted, varying)
        n_pred = predicted.sum()
        print(f"  {knob_label} vs {br}: varying codes {list(vcodes)} -> kaon codes {kcodes}; "
              f"predicted {n_pred:,} vs varying {varying.sum():,}; exact match: {exact}")
        ok_any = ok_any or exact
    return ok_any

record("kminus NEW: varying set == K- parent events (MCFlux ptype/tptype)",
       parent_crosscheck(varying_new, "kminus"))


## 5. kplus & kzero: new 1000-universe multisims

Single chunked pass over the NEW file accumulating per-event statistics, pathology counters, and
per-universe reco-$E_\nu$ histograms (weighted by the capped CV net weight, cap conventions
identical to `src/systematics.py`). The same pass also accumulates **true-$E_\nu$ histograms
split by true flavor** ($\nu_e$ vs $\nu_\mu$), used in section 7. Then the healthy-multisim
checks, kaon-parent cross-check, and an OLD-vs-NEW comparison of the kplus central values.

In [ ]:
netw_new = cap_weights(np.asarray(ev_new["weight_cv"], dtype=np.float64)
                       * np.asarray(ev_new["weight_spline"], dtype=np.float64))
inr_new = (np.asarray(enu_new) >= BINS[0]) & (np.asarray(enu_new) < BINS[-1])
bidx_new = np.digitize(np.asarray(enu_new)[inr_new], BINS) - 1
nominal_new = np.bincount(bidx_new, weights=netw_new[inr_new], minlength=NB)

NUNIV = {}
for c in [KPLUS, KZERO]:
    NUNIV[c] = int(ak.num(sw_new[c].array(entry_stop=1, library="ak"))[0])
print("universe counts (NEW):", NUNIV)

# true-flavor selections in TRUE Enu (flux uncertainties live in true energy) -- section 7
tr_new = f_new["wcpselection/T_eval"].arrays(["truth_nuPdg", "truth_nuEnergy"], library="np")
tenu_new = np.asarray(tr_new["truth_nuEnergy"], dtype=np.float64)
FLAV_NEW = {"nue": np.abs(tr_new["truth_nuPdg"]) == 12,
            "numu": np.abs(tr_new["truth_nuPdg"]) == 14}
TBINS = np.linspace(0, 3000, 16)   # MeV, true Enu (wider bins: only ~3k nue events)
TNB = len(TBINS) - 1
TBINC = 0.5 * (TBINS[:-1] + TBINS[1:])
tinr_new = (tenu_new >= TBINS[0]) & (tenu_new < TBINS[-1])
nominal_true = {t: np.bincount(np.digitize(tenu_new[s & tinr_new], TBINS) - 1,
                               weights=netw_new[s & tinr_new], minlength=TNB)
                for t, s in FLAV_NEW.items()}

ms = {c: dict(nonfinite=0, neg=0, sentinel=0, over_cap=0, wmax=-np.inf, wmin=np.inf,
              mean=np.zeros(N_new, np.float32), std=np.zeros(N_new, np.float32),
              varying=np.zeros(N_new, bool), hists=np.zeros((NUNIV[c], NB)),
              hists_nue=np.zeros((NUNIV[c], TNB)), hists_numu=np.zeros((NUNIV[c], TNB)))
      for c in [KPLUS, KZERO]}

start = 0
for chunk in sw_new.iterate([KPLUS, KZERO], step_size="600 MB", library="ak"):
    n = len(chunk[KPLUS])
    sl = slice(start, start + n)
    rmask = inr_new[sl]
    onehot = np.zeros((int(rmask.sum()), NB))
    if rmask.any():
        onehot[np.arange(rmask.sum()),
               np.digitize(np.asarray(enu_new)[sl][rmask], BINS) - 1] = netw_new[sl][rmask]
    fl_oh = {}
    for t, s in FLAV_NEW.items():
        mk = s[sl] & tinr_new[sl]
        o = np.zeros((int(mk.sum()), TNB))
        if mk.any():
            o[np.arange(mk.sum()), np.digitize(tenu_new[sl][mk], TBINS) - 1] = netw_new[sl][mk]
        fl_oh[t] = (mk, o)
    for c in [KPLUS, KZERO]:
        W = np.asarray(ak.flatten(chunk[c])).reshape(n, NUNIV[c])
        st = ms[c]
        finite = np.isfinite(W)
        st["nonfinite"] += int((~finite).sum())
        with np.errstate(invalid="ignore"):
            st["neg"] += int((W < 0).sum())
            st["sentinel"] += int((W < -1000).sum())
            st["over_cap"] += int((W > CAP).sum())
            if finite.any():
                st["wmax"] = max(st["wmax"], float(W[finite].max()))
                st["wmin"] = min(st["wmin"], float(W[finite].min()))
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            st["mean"][sl] = np.nanmean(W, axis=1)
            st["std"][sl] = np.nanstd(W, axis=1)
        st["varying"][sl] = ~np.all(W == W[:, :1], axis=1)
        Wc = cap_weights(W)
        st["hists"] += Wc[rmask].T @ onehot
        for t, (mk, o) in fl_oh.items():
            st["hists_" + t] += Wc[mk].T @ o
        del W, Wc
    start += n
    print(f"  processed {start:,}/{N_new:,} events", end="\r")
print()

for c in [KPLUS, KZERO]:
    st = ms[c]
    nv = int(st["varying"].sum())
    print(f"\n--- {c} ---")
    print(f"  varying events: {nv:,} ({nv / N_new:.2%})   "
          f"weight range [{st['wmin']:.4g}, {st['wmax']:.4g}]   >cap({CAP}): {st['over_cap']:,}")
    record(f"{c}: all weights finite", st["nonfinite"] == 0, f"{st['nonfinite']:,} non-finite")
    record(f"{c}: no -9898 sentinels", st["sentinel"] == 0, f"{st['sentinel']:,} sentinels")
    record(f"{c}: no negative weights", st["neg"] == 0, f"{st['neg']:,} negative")
    record(f"{c}: universes actually vary (bug was zero variation)", nv > 0)
    v = st["varying"]
    record(f"{c}: non-varying events have weight exactly 1",
           bool(np.all(st["mean"][~v] == 1.0) and np.all(st["std"][~v] == 0.0)))
    mu = float(np.mean(st["mean"][v])) if nv else float("nan")
    print(f"  mean over varying events of per-event universe-mean: {mu:.4f}")
    record(f"{c}: varying set == K parent events (MCFlux ptype/tptype)",
           parent_crosscheck(v, c.split('_')[0]))


In [ ]:
# distributions of the new multisim weights
fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
for c, ax in zip([KPLUS, KZERO], axs):
    v = ms[c]["varying"]
    ax.hist(ms[c]["mean"][v], bins=np.linspace(0, 3, 61), histtype="step",
            color=C_NEW, label="per-event mean")
    ax.hist(ms[c]["std"][v], bins=np.linspace(0, 3, 61), histtype="step",
            color="#444444", label="per-event std")
    ax.set_title(f"{c}\n({int(v.sum()):,} K-parent events)", fontsize=9)
    ax.set_xlabel("weight"); ax.set_yscale("log"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


In [ ]:
# kplus: the OLD file stored one identical value per event ("SciBooNE-constrained CV"?).
# Does the NEW multisim distribution center on that old value, or on 1?
KP_old = load_knob_matrix(sw_old, KPLUS)
old_identical = np.all(KP_old == KP_old[:, :1], axis=1)
record("kplus OLD: universes all identical per event (the original bug, for reference)",
       bool(old_identical.all()))
v_old_kp = KP_old[:, 0]
varying_old_kp = v_old_kp != 1.0
record("kplus: same events flagged K+ in OLD and NEW (matched events)",
       bool(np.array_equal(varying_old_kp[idx_old], ms[KPLUS]["varying"][idx_new])),
       f"OLD-matched {varying_old_kp[idx_old].sum():,} vs "
       f"NEW-matched {ms[KPLUS]['varying'][idx_new].sum():,}")

selv = ms[KPLUS]["varying"][idx_new] & varying_old_kp[idx_old]
x = v_old_kp[idx_old][selv]
y = ms[KPLUS]["mean"][idx_new][selv].astype(np.float64)
if selv.sum() > 1:
    corr = np.corrcoef(x, y)[0, 1]
    print(f"kplus matched K+ events: {selv.sum():,}; "
          f"OLD value mean {x.mean():.4f}, NEW universe-mean mean {y.mean():.4f}, corr {corr:.3f}")
    nshow = min(20000, selv.sum())
    step = max(1, selv.sum() // nshow)
    fig, ax = plt.subplots(figsize=(5.2, 5))
    ax.plot(x[::step], y[::step], ".", ms=2, alpha=0.25, color=C_NEW)
    lo, hi = min(x.min(), y.min()) - 0.05, max(x.max(), y.max()) + 0.05
    ax.plot([lo, hi], [lo, hi], "-", lw=1, color="#888888", label="y = x")
    ax.axhline(1.0, lw=0.8, ls=":", color="#888888")
    ax.set_xlabel("OLD kplus stored value (identical across 7 universes)")
    ax.set_ylabel("NEW kplus mean over 1000 universes")
    ax.set_title("kplus central value: OLD vs NEW (matched K+ events)", fontsize=10)
    ax.legend(); plt.tight_layout(); plt.show()

# kzero OLD sentinel recap
KZ_old = load_knob_matrix(sw_old, KZERO)
n_sent_old = int((KZ_old < -1000).sum())
print(f"kzero OLD for contrast: {n_sent_old:,} sentinel entries "
      f"({(np.any(KZ_old < -1000, axis=1)).sum():,} events all-sentinel)")


## 6. Impact on the reconstructed $E_\nu$ flux uncertainty

Per-universe reco-$E_\nu$ histograms with the standard analysis capping. OLD panels show the
pathology as the analysis would have seen it (capped); NEW panels show the fix. Then the per-bin
fractional 1σ from each kaon source, old vs new.

Prescriptions: multisims (kplus/kzero NEW) use the RMS of the 1000 universe histograms around the
nominal; the kminus multisigma uses the symmetrized $\pm1\sigma$ universes,
$(h_{+1}-h_{-1})/2$. OLD 7-universe knobs are shown with the RMS of their (capped) stored
universes — which is exactly the misleading content the old files carried.

In [ ]:
# OLD-file nominal + kaon-knob universe histograms (7 universes, cheap)
netw_old = cap_weights(np.asarray(ev_old["weight_cv"], dtype=np.float64)
                       * np.asarray(ev_old["weight_spline"], dtype=np.float64))
inr_old = (np.asarray(enu_old) >= BINS[0]) & (np.asarray(enu_old) < BINS[-1])
nominal_old = np.bincount(np.digitize(np.asarray(enu_old)[inr_old], BINS) - 1,
                          weights=netw_old[inr_old], minlength=NB)
onehot_old = np.zeros((int(inr_old.sum()), NB))
onehot_old[np.arange(inr_old.sum()),
           np.digitize(np.asarray(enu_old)[inr_old], BINS) - 1] = netw_old[inr_old]

hists_old = {}
for c, M in [(KMINUS, KM_old), (KPLUS, KP_old), (KZERO, KZ_old)]:
    hists_old[c] = cap_weights(M)[inr_old].T @ onehot_old

# NEW kminus universe histograms (5 universes)
onehot_new = np.zeros((int(inr_new.sum()), NB))
onehot_new[np.arange(inr_new.sum()), bidx_new] = netw_new[inr_new]
h_km_new = cap_weights(KM_new)[inr_new].T @ onehot_new

record("kminus NEW: sigma=0 universe histogram == nominal exactly",
       bool(np.allclose(h_km_new[1], nominal_new, rtol=0, atol=1e-9)))
del onehot_old, onehot_new; gc.collect();


In [ ]:
# per-universe ratio curves, OLD vs NEW
fig, axs = plt.subplots(3, 2, figsize=(11, 10), sharex=True)
SIG_NEW = [-1, 0, 1, 2, 3]
for irow, c in enumerate(KAON_KNOBS):
    axo, axn = axs[irow]
    for u in range(hists_old[c].shape[0]):
        axo.plot(BINC, hists_old[c][u] / nominal_old, drawstyle="steps-mid",
                 lw=1, color=C_OLD, alpha=0.7)
    axo.set_title(f"OLD (capped): {c}", fontsize=9)
    if c == KMINUS:
        for u in range(5):
            axn.plot(BINC, h_km_new[u] / nominal_new, drawstyle="steps-mid", lw=1.2,
                     color=C_NEW, alpha=0.8)
            axn.annotate(f"$\\sigma$={SIG_NEW[u]:+d}", (BINC[-1], (h_km_new[u] / nominal_new)[-1]),
                         fontsize=7, color=C_NEW, xytext=(3, 0), textcoords="offset points")
    else:
        h = ms[c]["hists"]
        r = h / nominal_new
        mean_r, rms_r = r.mean(axis=0), r.std(axis=0)
        axn.fill_between(BINC, mean_r - rms_r, mean_r + rms_r, step="mid",
                         color=C_NEW, alpha=0.25, label="mean ± RMS (1000 univ)")
        axn.plot(BINC, mean_r, drawstyle="steps-mid", lw=1.2, color=C_NEW)
        for u in range(0, NUNIV[c], 100):   # a faint sample of universes
            axn.plot(BINC, r[u], drawstyle="steps-mid", lw=0.4, color=C_NEW, alpha=0.15)
        axn.legend(fontsize=8)
    axn.set_title(f"NEW: {c}", fontsize=9)
    for ax in (axo, axn):
        ax.axhline(1, lw=0.8, color="#888888")
        ax.set_ylabel("universe / nominal")
axs[-1, 0].set_xlabel("reco $E_\\nu$ [MeV]"); axs[-1, 1].set_xlabel("reco $E_\\nu$ [MeV]")
plt.tight_layout(); plt.show()


In [ ]:
# per-bin fractional 1-sigma, old vs new
def frac_rms(h, nom):
    return np.sqrt(np.mean((h - nom[None, :]) ** 2, axis=0)) / np.where(nom > 0, nom, np.inf)

frac_new = {
    KMINUS: (h_km_new[2] - h_km_new[0]) / 2 / np.where(nominal_new > 0, nominal_new, np.inf),
    KPLUS: frac_rms(ms[KPLUS]["hists"], nominal_new),
    KZERO: frac_rms(ms[KZERO]["hists"], nominal_new),
}
frac_old = {c: frac_rms(hists_old[c], nominal_old) for c in KAON_KNOBS}
tot_new = np.sqrt(sum(frac_new[c] ** 2 for c in KAON_KNOBS))
tot_old = np.sqrt(sum(frac_old[c] ** 2 for c in KAON_KNOBS))

fig, axs = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
for ax, c in zip(axs, KAON_KNOBS):
    ax.plot(BINC, np.abs(frac_old[c]), drawstyle="steps-mid", color=C_OLD, label="OLD (capped)")
    ax.plot(BINC, np.abs(frac_new[c]), drawstyle="steps-mid", color=C_NEW, label="NEW")
    ax.set_title(c, fontsize=9); ax.set_xlabel("reco $E_\\nu$ [MeV]")
axs[0].set_ylabel("fractional 1$\\sigma$ on MC prediction")
axs[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(BINC, tot_old, drawstyle="steps-mid", color=C_OLD, lw=1.5,
        label="OLD total kaon flux (capped, broken)")
ax.plot(BINC, tot_new, drawstyle="steps-mid", color=C_NEW, lw=1.5, label="NEW total kaon flux")
ax.set_xlabel("reco $E_\\nu$ [MeV]"); ax.set_ylabel("fractional 1$\\sigma$")
ax.set_title("Total kaon-flux fractional uncertainty on reco $E_\\nu$", fontsize=10)
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

print(f"{'bin low edge':>12s} {'OLD total':>10s} {'NEW total':>10s}")
for i in range(NB):
    print(f"{BINS[i]:>12.0f} {tot_old[i]:>10.4f} {tot_new[i]:>10.4f}")


## 7. $\nu_e$ event weights and uncertainties

Intrinsic BNB $\nu_e$/$\bar\nu_e$ come mostly from $\mu^+$ decay at low energy and from
$K^+$/$K^0_L$ decay above ~1 GeV, so the kaon flux knobs matter far more for $\nu_e$ than for
the $\nu_\mu$ bulk. Here we split by true flavor (`truth_nuPdg`, all interactions CC+NC) and work
in **true** $E_\nu$ — flux uncertainties live in true energy, and the small $\nu_e$ sample is not
well served by WC reco $E_\nu$. We check the parentage composition, then compare the kaon-knob
fractional uncertainties on the $\nu_e$ prediction OLD vs NEW, and $\nu_e$ vs $\nu_\mu$.

Note the OLD panels are what the (capped) broken knobs would have claimed: kzero was all
sentinels → capped to exactly zero uncertainty, and kplus's identical universes produce a fake
flat "shift" instead of a spread — for the kaon-dominated $\nu_e$ flux these failures are much
more consequential than for $\nu_\mu$.

In [ ]:
# flavor composition and kaon parentage, nue vs numu
tr_old = f_old["wcpselection/T_eval"].arrays(["truth_nuPdg", "truth_nuEnergy"], library="np")
record("matched events: truth_nuPdg identical",
       bool(np.array_equal(np.asarray(tr_old["truth_nuPdg"])[idx_old],
                           np.asarray(tr_new["truth_nuPdg"])[idx_new])))

for tag, tr in [("OLD", tr_old), ("NEW", tr_new)]:
    pdg = np.asarray(tr["truth_nuPdg"])
    print(f"{tag}: " + "   ".join(f"{lbl} {np.sum(pdg == v):,}"
          for lbl, v in [("nue", 12), ("nuebar", -12), ("numu", 14), ("numubar", -14)]))

PT_NAMES = {-13: "mu+", 13: "mu-", 211: "pi+", -211: "pi-", 321: "K+", -321: "K-", 130: "K0L"}
ptype = np.asarray(fx_new["MCFlux_ptype"])
print("\nNEW direct parent (MCFlux_ptype) composition by true flavor:")
for t, s in FLAV_NEW.items():
    parts = sorted(((PT_NAMES.get(int(c), str(int(c))), float((ptype[s] == c).mean()))
                    for c in np.unique(ptype[s])), key=lambda x: -x[1])
    print(f"  {t:5s} ({int(s.sum()):,} events): "
          + "  ".join(f"{n0} {frv:.1%}" for n0, frv in parts))

# what the knobs actually act on: the per-knob varying sets (== target-primary kaon parentage)
kv = {KMINUS: varying_new, KPLUS: ms[KPLUS]["varying"], KZERO: ms[KZERO]["varying"]}
any_kaon = kv[KMINUS] | kv[KPLUS] | kv[KZERO]
print("\nfraction of events each kaon knob varies (NEW), by true flavor:")
fr = {}
for t, s in FLAV_NEW.items():
    fr[t] = float(any_kaon[s].mean())
    print(f"  {t:5s}: kplus {kv[KPLUS][s].mean():.1%}   kzero {kv[KZERO][s].mean():.1%}   "
          f"kminus {kv[KMINUS][s].mean():.2%}   any kaon {fr[t]:.1%}")
record("nue: kaon-ancestry fraction far exceeds numu's",
       fr["nue"] > 0.4 and fr["nue"] > 5 * fr["numu"],
       f"nue {fr['nue']:.1%} vs numu {fr['numu']:.1%}")

FLAV_LBL = {"nue": "$\\nu_e$", "numu": "$\\nu_\\mu$"}
fig, ax = plt.subplots(figsize=(6.5, 3.8))
for t, colr in [("nue", C_NEW), ("numu", "#444444")]:
    s = FLAV_NEW[t]
    den = np.histogram(tenu_new[s], bins=TBINS)[0].astype(float)
    num = np.histogram(tenu_new[s & any_kaon], bins=TBINS)[0]
    with np.errstate(invalid="ignore", divide="ignore"):
        ax.plot(TBINC, num / den, drawstyle="steps-mid", lw=1.5, color=colr,
                label=f"{FLAV_LBL[t]} ({int(s.sum()):,} events)")
ax.set_xlabel("true $E_\\nu$ [MeV]"); ax.set_ylabel("kaon-ancestry fraction")
ax.set_ylim(0, 1.05); ax.set_title("fraction of events touched by any kaon flux knob (NEW)",
                                   fontsize=10)
ax.legend(fontsize=9); plt.tight_layout(); plt.show()


In [ ]:
# kaon-knob fractional uncertainties on the nue prediction, true Enu: OLD vs NEW, and vs numu
tenu_old = np.asarray(tr_old["truth_nuEnergy"], dtype=np.float64)
tinr_old = (tenu_old >= TBINS[0]) & (tenu_old < TBINS[-1])
FLAV_OLD = {"nue": np.abs(np.asarray(tr_old["truth_nuPdg"])) == 12,
            "numu": np.abs(np.asarray(tr_old["truth_nuPdg"])) == 14}

def true_universe_hists(M, netw, tenu, tinr, sel):
    # (n_universes, TNB) capped-weight histograms in true Enu for events in sel;
    # M=None returns the nominal histogram
    mk = sel & tinr
    o = np.zeros((int(mk.sum()), TNB))
    o[np.arange(mk.sum()), np.digitize(tenu[mk], TBINS) - 1] = netw[mk]
    return o.sum(axis=0) if M is None else cap_weights(M)[mk].T @ o

nom_true_old, hists_true_old = {}, {}
for t, s in FLAV_OLD.items():
    nom_true_old[t] = true_universe_hists(None, netw_old, tenu_old, tinr_old, s)
    hists_true_old[t] = {c: true_universe_hists(M, netw_old, tenu_old, tinr_old, s)
                         for c, M in [(KMINUS, KM_old), (KPLUS, KP_old), (KZERO, KZ_old)]}

h_km_true = {t: true_universe_hists(KM_new, netw_new, tenu_new, tinr_new, s)
             for t, s in FLAV_NEW.items()}
for t in FLAV_NEW:
    record(f"kminus NEW ({t}): sigma=0 universe histogram == nominal exactly",
           bool(np.allclose(h_km_true[t][1], nominal_true[t], rtol=0, atol=1e-9)))

frac_true_new, frac_true_old, tot_true_new, tot_true_old = {}, {}, {}, {}
for t in FLAV_NEW:
    nz = np.where(nominal_true[t] > 0, nominal_true[t], np.inf)
    frac_true_new[t] = {
        KMINUS: np.abs(h_km_true[t][2] - h_km_true[t][0]) / 2 / nz,
        KPLUS: frac_rms(ms[KPLUS]["hists_" + t], nominal_true[t]),
        KZERO: frac_rms(ms[KZERO]["hists_" + t], nominal_true[t]),
    }
    frac_true_old[t] = {c: frac_rms(hists_true_old[t][c], nom_true_old[t]) for c in KAON_KNOBS}
    tot_true_new[t] = np.sqrt(sum(frac_true_new[t][c] ** 2 for c in KAON_KNOBS))
    tot_true_old[t] = np.sqrt(sum(frac_true_old[t][c] ** 2 for c in KAON_KNOBS))

fig, axs = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
for ax, c in zip(axs, KAON_KNOBS):
    ax.plot(TBINC, frac_true_old["nue"][c], drawstyle="steps-mid", color=C_OLD,
            label="OLD (capped, broken)")
    ax.plot(TBINC, frac_true_new["nue"][c], drawstyle="steps-mid", color=C_NEW, label="NEW")
    ax.set_title(c, fontsize=9); ax.set_xlabel("true $E_\\nu$ [MeV]")
axs[0].set_ylabel("fractional 1$\\sigma$ on $\\nu_e$ prediction")
axs[0].legend(fontsize=8)
fig.suptitle("kaon flux knobs on the true-$\\nu_e$ sample", fontsize=11)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot(TBINC, tot_true_old["nue"], drawstyle="steps-mid", color=C_OLD, lw=1.5,
        label="$\\nu_e$ OLD (capped, broken)")
ax.plot(TBINC, tot_true_new["nue"], drawstyle="steps-mid", color=C_NEW, lw=1.8,
        label="$\\nu_e$ NEW")
ax.plot(TBINC, tot_true_new["numu"], drawstyle="steps-mid", color=C_NEW, lw=1.2, ls="--",
        label="$\\nu_\\mu$ NEW (for scale)")
ax.set_xlabel("true $E_\\nu$ [MeV]"); ax.set_ylabel("fractional 1$\\sigma$")
ax.set_title("total kaon-flux uncertainty: $\\nu_e$ vs $\\nu_\\mu$, OLD vs NEW", fontsize=10)
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

pop = nominal_true["nue"] > 0
kdom = pop & (TBINC >= 1000)   # kaon-dominated region
record("nue NEW: kplus/kzero carry nonzero uncertainty in kaon-dominated bins (OLD had none real)",
       bool(np.all(frac_true_new["nue"][KPLUS][kdom] > 0)
            and np.all(frac_true_new["nue"][KZERO][kdom] > 0)))
both = kdom & (nominal_true["numu"] > 0)
record("nue NEW: total kaon uncertainty exceeds numu's in every kaon-dominated bin (>1 GeV)",
       bool(np.all(tot_true_new["nue"][both] > tot_true_new["numu"][both])))

print(f"{'bin low edge':>12s} {'nue OLD':>9s} {'nue NEW':>9s} {'numu NEW':>9s}   (nue nom. events)")
for i in range(TNB):
    n_ev = int(np.sum(FLAV_NEW['nue'] & tinr_new
                      & (np.digitize(tenu_new, TBINS) - 1 == i)))
    print(f"{TBINS[i]:>12.0f} {tot_true_old['nue'][i]:>9.4f} {tot_true_new['nue'][i]:>9.4f} "
          f"{tot_true_new['numu'][i]:>9.4f}   ({n_ev:,})")


## 8. Summary of all checks

In [ ]:
nwarn = 0
for name, status, detail in checks:
    if status == "WARN":
        nwarn += 1
    print(f"[{status}] {name}" + (f" — {detail}" if detail else ""))
print(f"\n{len(checks)} checks total, {nwarn} WARN")


## Notes / caveats

- The NEW file is a **partial test sample** (different subrun coverage than the full OLD
  production); the POT/event-count comparison in section 2 quantifies the fraction.
- **OLD-production POT bookkeeping (section 2b)**: the OLD file counts POT for thousands of
  subruns whose events are absent from its event trees (~7% of its POT), so its events-per-POT is
  correspondingly low; the NEW test sample does not have this problem. If MC is normalized by the
  summed `pot_tor875` of the OLD checkout, the Run 4b $\nu$ overlay prediction is biased low by
  that fraction — worth tracing where in the OLD retuple those events were lost.
- The handful of `weight_cv` mismatches are events where OLD stores exactly 1.0 while NEW stores
  a raw unphysical GENIE tune weight (negative or ≫1): the OLD retuple pre-sanitized bad
  `TunedCentralValue` events, the NEW raw checkout does not. The analysis capping
  (`postprocessing.py` / `src/systematics.py`) maps both to 1, so downstream results agree.
- `weightsReint` uses random throws, so it is only expected to be bit-identical if random seeds
  were preserved; otherwise statistical agreement is the passing condition.
- **Downstream code impact**: `src/create_rw_syst_df.py` and anything reading
  `spline_weights_df.parquet` classify knobs by universe count (e.g. "spline-morph knobs" =
  list columns with ≤50 universes, `weightsReint` special-cased at 1000). With this fix,
  `kplus_PrimaryHadronFeynmanScaling` and `kzero_PrimaryHadronSanfordWang` become 1000-wide
  **multisims** and must be moved out of the unisim/spline treatment (and PROfit spline inputs)
  into the multisim covariance machinery — and `kminus_PrimaryHadronNormalization` now has
  5 universes at $\sigma = [-1,0,+1,+2,+3]$, not 7 at $[-3\ldots+3]$, so any hard-coded
  $\sigma$ ordering must be updated.
- The uncertainty-impact sections use all events (no physics selection) with the capped CV net
  weight (`weight_cv * weight_spline`): section 6 in reco $E_\nu \in [0, 2000)$ MeV, section 7 in
  true $E_\nu \in [0, 3000)$ MeV — processing checks, not physics results.
- The $\nu_e$ sample is only ~3.3k events, so the section-7 histograms are MC-statistics limited
  bin-by-bin (counts printed in the table); the *fractional* uncertainties are still meaningful
  because every universe reweights the same events, but bin-to-bin jitter reflects which specific
  kaon-ancestry events populate each bin.